# Requêtes de contrôle NutriScope (TP4, jalon J1)

Cinq contrôles sur la base relationnelle : volumétrie par table, produits sans catégorie,
top marques, complétude Nutri-Score par rayon, doublons restants.

## Prérequis

Avant d'exécuter ce notebook, il faut avoir, dans l'ordre :

1. **Suivi [`docs/postgresql_install.md`](../docs/postgresql_install.md)** : conteneur Docker
   PostgreSQL démarré (`docker compose up -d` dans `Docker_SQL/`), base `nutriscope`
   accessible sur `localhost:5432`.
2. **Créé le schéma** : [`sql/schema.sql`](../sql/schema.sql) exécuté sur la base `nutriscope`
   (DBeaver, ou `docker exec -i postgres psql -U postgres -d nutriscope < sql/schema.sql`).
3. **Généré les extraits Parquet du périmètre** : `python src/extract_perimeter.py` (lit
   `data/food.parquet`, écrit `data/extracts/*.parquet`).
4. **Chargé la base** : `python src/load_parquet.py` (vide les tables puis recharge depuis
   `data/extracts/`).

Sans ces quatre étapes, les requêtes ci-dessous s'exécuteront sur une base vide ou absente.

Comme pour [`src/run_control_queries.py`](../src/run_control_queries.py), les requêtes passent
par `docker exec ... psql` plutôt que par une connexion directe (psycopg2/DuckDB) : ces
dernières plantent sur certaines machines Windows avec un `UnicodeDecodeError` au moment de la
connexion réseau (bug d'environnement lié à libpq, indépendant du projet). `docker exec` reste
à l'intérieur du conteneur et évite complètement ce problème.

In [8]:
import io
import subprocess

import pandas as pd

CONTAINER_NAME = "postgres"
DB_NAME = "nutriscope"
DB_USER = "postgres"


def run_query(sql: str) -> pd.DataFrame:
    """Exécute une requête SQL dans le conteneur Postgres et renvoie un DataFrame."""

    result = subprocess.run(
        [
            "docker", "exec", "-i", CONTAINER_NAME,
            "psql", "-U", DB_USER, "-d", DB_NAME,
            "--csv", "-c", sql,
        ],
        capture_output=True,
    )

    if result.returncode != 0:
        raise RuntimeError(result.stderr.decode("utf-8"))

    return pd.read_csv(io.StringIO(result.stdout.decode("utf-8")))

## 1. Volumétrie par table

In [9]:
run_query('''
    SELECT 'brands' AS table_name, COUNT(*) AS nb_lignes FROM brands
    UNION ALL
    SELECT 'categories', COUNT(*) FROM categories
    UNION ALL
    SELECT 'products', COUNT(*) FROM products
    UNION ALL
    SELECT 'products_categories', COUNT(*) FROM products_categories
    UNION ALL
    SELECT 'nutrients', COUNT(*) FROM nutrients
    ORDER BY table_name;
''')

,table_name,nb_lignes
0,brands,79577
1,categories,37442
2,nutrients,1247309
3,products,1247309
4,products_categories,3939635


## 2. Produits sans catégorie

In [10]:
run_query('''
    SELECT COUNT(*) AS nb_produits_sans_categorie
    FROM products AS p
    WHERE NOT EXISTS (
        SELECT 1 FROM products_categories AS pc WHERE pc.code = p.code
    );
''')

,nb_produits_sans_categorie
0,630068


## 3. Top marques (par nombre de produits)

In [11]:
run_query('''
    SELECT b.name AS marque, COUNT(*) AS nb_produits
    FROM products AS p
    JOIN brands AS b ON b.id = p.brand_id
    GROUP BY b.name
    ORDER BY nb_produits DESC
    LIMIT 10;
''')

,marque,nb_produits
0,xx:carrefour,11277
1,xx:u,10749
2,xx:auchan,8164
3,xx:casino,6225
4,xx:marque-repere,6217
5,xx:nestle,5898
6,xx:leader-price,5690
7,xx:U,5225
8,xx:le-gaulois,4478
9,xx:cora,4034


## 4. Complétude Nutri-Score par rayon

Le mapping catégories OFF -> 6 rayons n'est pas encore tranché définitivement
(cf. [`docs/perimetre.md`](../docs/perimetre.md), risque R-D1 dans
[`docs/cadrage/donnees.md`](../docs/cadrage/donnees.md)). Le mapping ci-dessous est provisoire,
basé sur les tags OFF les plus fréquents du périmètre France, utilisé uniquement pour ce
contrôle de complétude.

In [12]:
run_query('''
    WITH rayon AS (
        SELECT
            pc.code,
            CASE
                WHEN c.tag = 'en:beverages' THEN 'Boissons'
                WHEN c.tag = 'en:dairies' THEN 'Produits laitiers'
                WHEN c.tag = 'en:breakfasts' THEN 'Céréales et petit-déjeuner'
                WHEN c.tag IN ('en:biscuits-and-cakes', 'en:salty-snacks') THEN 'Biscuits et snacks'
                WHEN c.tag IN ('en:meals', 'en:canned-foods') THEN 'Plats préparés et conserves'
                WHEN c.tag IN ('en:sauces', 'en:condiments') THEN 'Sauces et condiments'
            END AS rayon
        FROM products_categories AS pc
        JOIN categories AS c ON c.id = pc.category_id
    ),
    produit_rayon AS (
        -- un produit peut porter plusieurs tags du même rayon (ou de rayons différents) :
        -- on ne le compte qu'une fois par rayon distinct
        SELECT DISTINCT code, rayon FROM rayon WHERE rayon IS NOT NULL
    )
    SELECT
        pr.rayon,
        COUNT(*) AS nb_produits,
        COUNT(*) FILTER (WHERE p.nutriscore_grade IN ('a','b','c','d','e')) AS nb_avec_nutriscore,
        ROUND(
            100.0 * COUNT(*) FILTER (WHERE p.nutriscore_grade IN ('a','b','c','d','e')) / COUNT(*),
            1
        ) AS pct_complet
    FROM produit_rayon AS pr
    JOIN products AS p ON p.code = pr.code
    GROUP BY pr.rayon
    ORDER BY pct_complet DESC;
''')

,rayon,nb_produits,nb_avec_nutriscore,pct_complet
0,Biscuits et snacks,55038,50193,91.2
1,Plats préparés et conserves,60005,53843,89.7
2,Produits laitiers,59861,50230,83.9
3,Céréales et petit-déjeuner,33298,22028,66.2
4,Sauces et condiments,37443,23641,63.1
5,Boissons,70702,36861,52.1


## 5. Doublons restants

In [13]:
run_query('''
    SELECT COUNT(*) AS nb_doublons_stricts
    FROM (SELECT code FROM products GROUP BY code HAVING COUNT(*) > 1) AS d;
''')

,nb_doublons_stricts
0,0
